# 🚀 Módulo 5: Despliegue de Modelos de Machine Learning
## 18. Creando una API con FastAPI y Contenerizando con Docker

### Curso: **Machine Learning con Python** (IFCD093PO)
**Duración estimada:** 6 horas

---

## 🎯 Objetivos del Notebook

En este notebook, pasaremos de la teoría a la práctica. Vamos a tomar uno de los modelos que construimos en los proyectos finales (el de **regresión para estimar el precio de alquileres**) y lo desplegaremos como un servicio web utilizando herramientas estándar de la industria.

**Objetivos principales:**
1.  **Preparar el Modelo**: Cargar el pipeline de preprocesamiento y el modelo de regresión que guardamos previamente.
2.  **Construir una API REST con FastAPI**: Crear los endpoints necesarios para recibir datos, procesarlos, obtener una predicción y devolverla.
3.  **Definir los Modelos de Datos con Pydantic**: Asegurar que los datos de entrada y salida de nuestra API tengan la estructura correcta.
4.  **Contenerizar la Aplicación con Docker**: Crear un `Dockerfile` para empaquetar nuestra API y todas sus dependencias en una imagen de Docker, haciéndola portable y escalable.
5.  **Probar la API**: Levantar el contenedor y enviar peticiones para verificar que todo funciona correctamente.

**Aclaración:** Gran parte del trabajo se realizará fuera de este notebook, en archivos `.py` y un `Dockerfile`. Este notebook servirá como guía y para ejecutar los comandos necesarios.

## 📚 Flujo de Trabajo

1. [Revisión de la Estructura del Proyecto](#1-estructura)
2. [Paso 1: Preparar y Guardar el Modelo y el Preprocesador](#2-guardar-modelo)
3. [Paso 2: Crear la Aplicación con FastAPI (`main.py`)](#3-fastapi)
   - [Instalación de Librerías](#3.1-install)
   - [Definición de los Modelos de Datos con Pydantic](#3.2-pydantic)
   - [Creación de la App y los Endpoints](#3.3-endpoints)
4. [Paso 3: Crear el `Dockerfile`](#4-dockerfile)
5. [Paso 4: Construir y Ejecutar el Contenedor Docker](#5-build-run)
   - [Construir la Imagen (`docker build`)](#5.1-build)
   - [Ejecutar el Contenedor (`docker run`)](#5.2-run)
6. [Paso 5: Probar la API en Producción](#6-probar)
   - [Usando la Documentación Interactiva de FastAPI](#6.1-docs)
   - [Usando un Cliente de Python (`requests`)](#6.2-requests)
7. [Resumen y Próximos Pasos](#7-resumen)

---

## 1. Revisión de la Estructura del Proyecto <a id='1-estructura'></a>

Para este despliegue, organizaremos nuestro código en una estructura de carpetas clara. Crearemos una nueva carpeta llamada `api_alquiler/` con la siguiente estructura:

```
api_alquiler/
├── app/
│   ├── __init__.py
│   ├── main.py           # Lógica de la API con FastAPI
│   ├── models.py         # Modelos de datos con Pydantic
│   └── assets/
│       ├── preprocessor.joblib # Nuestro preprocesador guardado
│       └── model.joblib        # Nuestro modelo de regresión guardado
├── Dockerfile            # Instrucciones para construir la imagen Docker
└── requirements.txt      # Dependencias de Python
```

Este notebook nos guiará para crear cada uno de estos archivos.

---

## 2. Paso 1: Preparar y Guardar el Modelo y el Preprocesador <a id='2-guardar-modelo'></a>

Primero, necesitamos volver al notebook `15_proyecto_regresion.ipynb`, re-entrenar el pipeline completo (preprocesador + modelo) con **todos los datos** y guardar las dos partes por separado: el `preprocessor` y el `regressor`.

**Acción**: Añade una celda al final del notebook `15` para guardar estos dos componentes usando `joblib`.

In [ ]:
# Este código NO se ejecuta aquí. Es un EJEMPLO de lo que deberías añadir
# al final del notebook 15_proyecto_regresion.ipynb para guardar los artefactos.

# --- CÓDIGO PARA EL NOTEBOOK 15 ---
import joblib # Para guardar el modelo
import os # Para manejo de rutas

# Suponiendo que 'best_pipeline' es tu pipeline final entrenado
final_model = best_pipeline.named_steps['regressor']
preprocessor = best_pipeline.named_steps['preprocessor'] 

# Crear la carpeta de assets si no existe
os.makedirs('api_alquiler/app/assets', exist_ok=True)

# Guardar el modelo y el preprocesador
joblib.dump(final_model, 'api_alquiler/app/assets/model.joblib')
joblib.dump(preprocessor, 'api_alquiler/app/assets/preprocessor.joblib')

print("Modelo y preprocesador guardados exitosamente.")

---

## 3. Paso 2: Crear la Aplicación con FastAPI (`main.py`) <a id='3-fastapi'></a>

Ahora crearemos los archivos Python que definirán nuestra API.

### 3.1. Instalación de Librerías y `requirements.txt` <a id='3.1-install'></a>

Necesitaremos algunas librerías. Las instalamos y las guardamos en `requirements.txt`.

In [ ]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib pandas numpy
# Guardar las dependencias en requirements.txt
!pip freeze > api_alquiler/requirements.txt

### 3.2. `app/models.py`: Definición de los Modelos de Datos con Pydantic <a id='3.2-pydantic'></a>

Pydantic nos permite definir la estructura de los datos de entrada y salida. FastAPI lo usa para validar automáticamente las peticiones.

**Acción**: Crea el archivo `api_alquiler/app/models.py` con el siguiente contenido.

In [ ]:
# Contenido para api_alquiler/app/models.py
# --- CÓDIGO PARA api_alquiler/app/models.py ---
# Explicación del código:
# Este archivo define los modelos de datos utilizando Pydantic.
# Se crean dos clases: PropertyFeatures para las características de la propiedad
# y PredictionOutput para la salida de la predicción del valor del alquiler.
from pydantic import BaseModel # Para definir los modelos de datos
from typing import Literal # Para tipos literales en Pydantic

class PropertyFeatures(BaseModel):
    city: Literal['São Paulo', 'Porto Alegre', 'Rio de Janeiro', 'Campinas', 'Belo Horizonte']
    area: int
    rooms: int
    bathroom: int
    parking_spaces: int
    animal: Literal['acept', 'not acept']
    furniture: Literal['furnished', 'not furnished']

    class Config:
        schema_extra = {
            "example": {
                "city": "São Paulo",
                "area": 150,
                "rooms": 3,
                "bathroom": 2,
                "parking_spaces": 2,
                "animal": "acept",
                "furniture": "furnished"
            }
        }

class PredictionOutput(BaseModel):
    predicted_rent_value: float

### 3.3. `app/main.py`: Creación de la App y los Endpoints <a id='3.3-endpoints'></a>

Este es el corazón de nuestra API. Cargará el modelo y definirá la lógica para las predicciones.

**Acción**: Crea el archivo `api_alquiler/app/main.py` con el siguiente contenido.

In [ ]:
# Contenido para api_alquiler/app/main.py
# --- CÓDIGO PARA api_alquiler/app/main.py ---
# Explicación del código:
# Este archivo contiene la implementación de la API utilizando FastAPI.
# Se define un endpoint de salud y un endpoint de predicción que recibe
# las características de la propiedad y devuelve el valor predicho del alquiler.

from fastapi import FastAPI
from .models import PropertyFeatures, PredictionOutput
import joblib
import pandas as pd
import numpy as np

# Crear la instancia de FastAPI
app = FastAPI(title="API de Predicción de Alquileres", version="1.0")

# Cargar modelo y preprocesador
model = joblib.load('app/assets/model.joblib')
preprocessor = joblib.load('app/assets/preprocessor.joblib')

# Endpoint de salud, se usa para verificar que la API está funcionando
@app.get("/", tags=["Health Check"])
def health_check():
    return {"status": "OK"}

# Endpoint de predicción, recibe las características de la propiedad y devuelve el valor predicho del alquiler
# Parametros post:
#  features: PropertyFeatures - características de la propiedad
# Retorna:
#  PredictionOutput - valor predicho del alquiler
@app.post("/predict", response_model=PredictionOutput, tags=["Predictions"])

# Función para realizar la predicción
def predict(features: PropertyFeatures):
    # Convertir los datos de entrada a un DataFrame de pandas
    input_data = pd.DataFrame([features.dict()])
    
    # Preprocesar los datos
    processed_data = preprocessor.transform(input_data)
    
    # Realizar la predicción (el modelo predice el logaritmo del precio)
    log_prediction = model.predict(processed_data)
    
    # Revertir la transformación logarítmica para obtener el valor real
    prediction = np.expm1(log_prediction)[0]
    
    return {"predicted_rent_value": round(prediction, 2)}

---

## 4. Paso 3: Crear el `Dockerfile` <a id='4-dockerfile'></a>

El `Dockerfile` es una receta que le dice a Docker cómo construir la imagen de nuestra aplicación. Define el sistema operativo base, copia nuestros archivos, instala las dependencias y especifica el comando para ejecutar la API.

**Acción**: Crea el archivo `api_alquiler/Dockerfile` con el siguiente contenido.

In [ ]:
# Contenido para api_alquiler/Dockerfile
# --- CÓDIGO PARA api_alquiler/Dockerfile ---
# Explicación del código:
# Este Dockerfile crea una imagen de Docker para la API de FastAPI.
# Se utiliza una imagen base de Python, se copian los archivos necesarios,
# se instalan las dependencias y se expone el puerto para la API.

# 1. Usar una imagen base oficial de Python
FROM python:3.10-slim

# 2. Establecer el directorio de trabajo dentro del contenedor
WORKDIR /code

# 3. Copiar el archivo de requerimientos e instalar dependencias
COPY ./requirements.txt /code/requirements.txt
RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt

# 4. Copiar el resto de la aplicación (la carpeta app)
COPY ./app /code/app

# 5. Exponer el puerto en el que correrá la API
EXPOSE 8000

# 6. Comando para ejecutar la aplicación usando uvicorn
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

---

## 5. Paso 4: Construir y Ejecutar el Contenedor Docker <a id='5-build-run'></a>

Con todos los archivos en su sitio, ahora podemos usar Docker. Asegúrate de tener Docker Desktop instalado y en ejecución.

### 5.1. Construir la Imagen (`docker build`) <a id='5.1-build'></a>

Este comando lee el `Dockerfile` y crea una imagen llamada `rent-predictor-api`.

In [ ]:
# Ejecuta este comando en tu terminal, desde la carpeta raíz 'api_alquiler/'
!docker build -t rent-predictor-api .

### 5.2. Ejecutar el Contenedor (`docker run`) <a id='5.2-run'></a>

Este comando inicia un contenedor a partir de la imagen que acabamos de crear. Mapea el puerto 8000 del contenedor al puerto 8000 de tu máquina local.

In [ ]:
# Ejecuta este comando en tu terminal
!docker run -d -p 8000:8000 --name rent_api_container rent-predictor-api

---

## 6. Paso 5: Probar la API en Producción <a id='6-probar'></a>

¡Nuestra API ya está corriendo dentro de un contenedor Docker!

### 6.1. Usando la Documentación Interactiva de FastAPI <a id='6.1-docs'></a>

Abre tu navegador y ve a **`http://localhost:8000/docs`**.

Verás la documentación autogenerada por FastAPI. Puedes desplegar el endpoint `/predict`, hacer clic en "Try it out", rellenar el ejemplo y ejecutar una predicción directamente desde el navegador.

### 6.2. Usando un Cliente de Python (`requests`) <a id='6.2-requests'></a>

También podemos probarlo mediante programación.

In [ ]:
# Contenido para test_api.py
# --- CÓDIGO PARA test_api.py ---
# Explicación del código:
# Este script prueba el endpoint de predicción de la API.
# Envía una solicitud POST con datos de ejemplo y muestra la respuesta.
import requests
import json

url = "http://localhost:8000/predict"

data = {
    "city": "São Paulo",
    "area": 200,
    "rooms": 4,
    "bathroom": 3,
    "parking_spaces": 2,
    "animal": "acept",
    "furniture": "furnished"
}

response = requests.post(url, data=json.dumps(data))

if response.status_code == 200:
    print("Petición exitosa!")
    print("Respuesta:", response.json())
else:
    print(f"Error: {response.status_code}")
    print("Respuesta:", response.text)

---

## 7. Resumen y Próximos Pasos <a id='7-resumen'></a>

¡Felicidades! Has desplegado con éxito un modelo de Machine Learning como una API robusta y portable.

✅ Has aprendido a estructurar un proyecto de despliegue.

✅ Has creado una API REST con FastAPI para servir tu modelo.

✅ Has empaquetado toda la aplicación en un contenedor Docker.

✅ Has probado tu API en un entorno de producción simulado.

Este es un hito fundamental en el camino de un científico de datos. A partir de aquí, los siguientes pasos en un entorno real serían:

- **Subir la imagen de Docker a un registro de contenedores** (como Docker Hub, AWS ECR, Google GCR).
- **Desplegar el contenedor en un servicio en la nube** (como AWS ECS, Google Cloud Run, o un clúster de Kubernetes).
- **Configurar un pipeline de CI/CD** para automatizar las actualizaciones.